# LiH Adaptive VQE (AdaptVQE)

This notebook implements an adaptive UCCSD-style VQE for LiH using the current verified Qiskit stack.

The key change from the previous notebook is **AdaptVQE**: it starts from the Hartree–Fock state and adds one excitation at a time instead of optimizing the full UCCSD ansatz immediately.


## Installation

Run the first lines of the code cell below once, then restart the Jupyter kernel.

Versions verified on October 1, 2026: Qiskit 2.5.2, Qiskit Nature 0.8.0, Qiskit Algorithms 0.4.0, and PySCF 2.14.0.


In [1]:

# LiH Adaptive VQE (AdaptVQE)
# Verified stack: Qiskit 2.5.2, Qiskit Nature 0.8.0,
# Qiskit Algorithms 0.4.0, PySCF 2.14.0.

# ------------------------------------------------------------
# 0. Install dependencies
# ------------------------------------------------------------
# Run this once in qBraid/Jupyter, then RESTART the kernel.
#
# %pip install -U \
#     "qiskit==2.5.2" \
#     "qiskit-nature==0.8.0" \
#     "qiskit-algorithms==0.4.0" \
#     "pyscf==2.14.0"

# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------
import time
import warnings
import numpy as np
from importlib.metadata import version

from scipy.sparse import SparseEfficiencyWarning

# These SciPy warnings are performance notices, not errors.
warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)

from qiskit.primitives import StatevectorEstimator

from qiskit_algorithms import (
    VQE,
    AdaptVQE,
    NumPyMinimumEigensolver,
)
from qiskit_algorithms.optimizers import SLSQP

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import FreezeCoreTransformer
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.algorithms import GroundStateEigensolver



# ------------------------------------------------------------
# 3. Molecular problem: LiH / STO-3G
# ------------------------------------------------------------
BOND_LENGTH = 1.595  # Angstrom

driver = PySCFDriver(
    atom=f"Li 0 0 0; H 0 0 {BOND_LENGTH}",
    unit=DistanceUnit.ANGSTROM,
    basis="sto3g",
    charge=0,
    spin=0,
)

problem = driver.run()

print("Before frozen core")
print("------------------")
print(f"Spatial orbitals : {problem.num_spatial_orbitals}")
print(f"Particles        : {problem.num_particles}")
print()

# Freeze the Li 1s core.
problem = FreezeCoreTransformer().transform(problem)

print("After frozen core")
print("-----------------")
print(f"Spatial orbitals : {problem.num_spatial_orbitals}")
print(f"Particles        : {problem.num_particles}")
print(f"Active spin orbitals: {2 * problem.num_spatial_orbitals}")
print(f"Nuclear repulsion: {problem.nuclear_repulsion_energy:.10f} Ha")
print()


# ------------------------------------------------------------
# 4. Qubit mapping
# ------------------------------------------------------------
# ParityMapper with num_particles exploits particle-number symmetry
# and reduces the qubit count compared with plain Jordan-Wigner.
mapper = ParityMapper(num_particles=problem.num_particles)

hf_state = HartreeFock(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    qubit_mapper=mapper,
)

print(f"Qubits after parity reduction: {hf_state.num_qubits}")
print()


# ------------------------------------------------------------
# 5. Build the UCCSD excitation pool
# ------------------------------------------------------------
# The pool contains all single and double excitation operators.
# AdaptVQE will NOT optimize all of them simultaneously.
uccsd_pool = UCCSD(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    qubit_mapper=mapper,
)

pool = uccsd_pool.operators

print(f"UCCSD excitation-pool size: {len(pool)}")
print()


# ------------------------------------------------------------
# 6. Inner VQE
# ------------------------------------------------------------
estimator = StatevectorEstimator()

evaluation_history = []
best_electronic_energy = np.inf


def vqe_callback(eval_count, parameters, mean, metadata):
    """Record and print the current and best VQE objective value."""
    global best_electronic_energy

    energy = float(np.real(mean))
    evaluation_history.append(energy)
    best_electronic_energy = min(best_electronic_energy, energy)

    if eval_count == 1 or eval_count % 10 == 0:
        print(
            f"eval {eval_count:4d} | "
            f"current = {energy:.10f} Ha | "
            f"best = {best_electronic_energy:.10f} Ha"
        )


# Each AdaptVQE iteration only optimizes the small ansatz constructed
# up to that point. SLSQP is therefore much more practical here than
# using a large fixed UCCSD ansatz from the beginning.
optimizer = SLSQP(
    maxiter=50,
    ftol=1e-8,
)

vqe = VQE(
    estimator=estimator,
    ansatz=hf_state,
    optimizer=optimizer,
    callback=vqe_callback,
)


# ------------------------------------------------------------
# 7. Adaptive VQE
# ------------------------------------------------------------
adapt_vqe = AdaptVQE(
    vqe,
    operators=pool,
    initial_state=hf_state,
    gradient_threshold=1e-5,
    eigenvalue_threshold=1e-6,
    max_iterations=10,
)

print("Starting AdaptVQE")
print("=================")

start = time.perf_counter()

result = GroundStateEigensolver(
    mapper,
    adapt_vqe,
).solve(problem)

elapsed = time.perf_counter() - start

# ------------------------------------------------------------
# 8. VQE result
# ------------------------------------------------------------
vqe_total_energy = float(np.real(result.total_energies[0]))

raw = result.raw_result

print()
print("========== ADAPTVQE RESULT ==========")
print(f"Final total energy       : {vqe_total_energy:.10f} Ha")
print(f"Runtime                  : {elapsed:.2f} s")
print(f"Adaptive iterations      : {raw.num_iterations}")
print(f"Final maximum gradient   : {abs(raw.final_max_gradient):.6e}")
print(f"Termination criterion    : {raw.termination_criterion}")
print()


# ------------------------------------------------------------
# 9. Exact reference
# ------------------------------------------------------------
print("Computing exact reference...")

exact_solver = GroundStateEigensolver(
    mapper,
    NumPyMinimumEigensolver(),
)

exact_result = exact_solver.solve(problem)

exact_total_energy = float(np.real(exact_result.total_energies[0]))

error = abs(vqe_total_energy - exact_total_energy)
chemical_accuracy = 1.6e-3  # Hartree = 1.6 mHa

print()
print("========== FINAL COMPARISON ==========")
print(f"Exact total energy       : {exact_total_energy:.10f} Ha")
print(f"AdaptVQE total energy    : {vqe_total_energy:.10f} Ha")
print(f"Absolute error           : {error:.6e} Ha")
print(f"Absolute error           : {error * 1000:.4f} mHa")
print(f"Chemical accuracy        : {error < chemical_accuracy}")
print()


# ------------------------------------------------------------
# 10. Adaptive convergence history
# ------------------------------------------------------------
print("========== ADAPTIVE HISTORY ==========")
print("Iteration | Electronic eigenvalue (Ha)")
print("----------|----------------------------")

for i, energy in enumerate(raw.eigenvalue_history, start=1):
    print(f"{i:9d} | {float(np.real(energy)):.10f}")


# ------------------------------------------------------------
# 11. Basic sanity checks
# ------------------------------------------------------------
print()
print("========== SANITY CHECKS ==========")

if vqe_total_energy > exact_total_energy + 1e-8:
    print(
        "VQE energy is above the exact energy, as expected for an "
        "approximate variational solution."
    )
else:
    print(
        "The VQE result is numerically at or below the exact reference. "
        "Check numerical tolerances/definitions if this occurs."
    )

if error < chemical_accuracy:
    print("PASS: VQE is within chemical accuracy.")
else:
    print(
        "NOT YET WITHIN CHEMICAL ACCURACY. "
        "Try increasing max_iterations or tightening the inner optimizer."
    )


Installed versions
------------------
qiskit              : 2.5.0      [CHECK]
qiskit-nature       : 0.8.0      [OK]
qiskit-algorithms   : 0.4.0      [OK]
pyscf               : 2.13.1     [CHECK]

Before frozen core
------------------
Spatial orbitals : 6
Particles        : (2, 2)

After frozen core
-----------------
Spatial orbitals : 5
Particles        : (1, 1)
Active spin orbitals: 10
Nuclear repulsion: 0.9953176381 Ha

Qubits after parity reduction: 8

UCCSD excitation-pool size: 24

Starting AdaptVQE
eval    1 | current = -1.0590503101 Ha | best = -1.0590503101 Ha
eval    1 | current = -1.0735630630 Ha | best = -1.0735630630 Ha
eval   10 | current = -1.0754606685 Ha | best = -1.0754609425 Ha
eval    1 | current = -1.0754619379 Ha | best = -1.0754619379 Ha
eval   10 | current = -1.0772189408 Ha | best = -1.0772189408 Ha
eval   20 | current = -1.0772204683 Ha | best = -1.0772204683 Ha
eval    1 | current = -1.0772204810 Ha | best = -1.0772204810 Ha
eval   10 | current = -1.077707860